# CNN model

In [1]:
import pathlib
import keras
from keras import layers
from keras.utils import image_dataset_from_directory
import numpy as np

<h2>1. Business understanding</h2>

   - Määritellään ratkaistava liiketoimintaongelma (business problem).
   - Tavoitteena on selvittää analyysin päämäärä (goal), vaatimukset (requirements), rajoitteet (constraints) ja odotettu lopputulos (expected outcome).

<h2>2. Data understanding</h2>

   - Kerätään ja tutkitaan saatavilla olevaa dataa.
   - Tässä kartoitetaan datan ominaisuudet, kuten muuttujatyypit ja arvojakaumat, sekä mahdolliset laatuongelmat (quality issues), kuten puuttuvat arvot (missing values), poikkeamat (outliers) ja järjettömät arvot (nonsensical values).

<h2>3. Data preparation</h2>

   - Data siivotaan, muunnetaan ja valitaan olennaiset piirteet (features).
   - Kaikki nämä vaiheet tulee dokumentoida niin tarkasti, että ne ovat myöhemmin täysin toistettavissa.

Load the datasets from the directory. English and Finnish folders are loaded separately in order to get correct class names.

In [2]:
# load dataset from directory

data_dir = pathlib.Path("../data/image-samples") # directory with all dataS

# English datasets
en_train_dataset = image_dataset_from_directory(
    data_dir / "train" / "english",
    image_size = (224, 224),
    batch_size = 32)

en_validation_dataset = image_dataset_from_directory(
    data_dir / "validation" / "english",
    image_size = (224, 224),
    batch_size = 32)

en_test_dataset = image_dataset_from_directory(
    data_dir / "test" / "english",
    image_size = (224, 224),
    batch_size = 32,
    shuffle=False)

# Finnish datasets
fi_train_dataset = image_dataset_from_directory(
    data_dir / "train" / "finnish",
    image_size=(224, 224),
    batch_size=32)

fi_validation_dataset = image_dataset_from_directory(
    data_dir / "validation" / "finnish",
    image_size=(224, 224),
    batch_size=32)

fi_test_dataset = image_dataset_from_directory(
    data_dir / "test" / "finnish",
    image_size=(224, 224),
    batch_size=32,
    shuffle=False)

Found 116 files belonging to 6 classes.
Found 18 files belonging to 6 classes.
Found 18 files belonging to 6 classes.
Found 119 files belonging to 6 classes.
Found 18 files belonging to 6 classes.
Found 18 files belonging to 6 classes.


Let's verify that all of the class names are correct.

In [3]:
en_class_names = en_train_dataset.class_names
fi_class_names = fi_train_dataset.class_names

print("English class names:", en_class_names)
print("Finnish class names:", fi_class_names)

English class names: ['down', 'left', 'right', 'start', 'stop', 'up']
Finnish class names: ['alas', 'oikea', 'paalle', 'pysahdy', 'vasen', 'ylos']


That's correct. We have 6 English and 6 Finnish classes, a total of 12 classes. `image_dataset_from_directory()` has converted the class names into integer values, let's check those for each dataset.

In [4]:
def check_labels(dataset, name):
    all_labels = np.concatenate([
        labels.numpy() for images, labels in dataset
    ])

    print(f"\n{name}")
    print("Unique labels:", np.unique(all_labels))

check_labels(en_train_dataset, "English train dataset")
check_labels(fi_train_dataset, "Finnish train dataset")


English train dataset
Unique labels: [0 1 2 3 4 5]

Finnish train dataset
Unique labels: [0 1 2 3 4 5]


Now the English and Finnish class indices are overlapping each other. Let's fix the indices of the Finnish classes.

In [5]:
def shift_labels(images, labels):
    return images, labels + 6

fi_train_dataset = fi_train_dataset.map(shift_labels)
fi_validation_dataset = fi_validation_dataset.map(shift_labels)
fi_test_dataset = fi_test_dataset.map(shift_labels)

In [6]:
def check_labels(dataset):
    all_labels = np.concatenate([
        labels.numpy() for images, labels in dataset
    ])
    print("Unique labels:", np.unique(all_labels))

print("English")
check_labels(en_train_dataset)

print("\nFinnish")
check_labels(fi_train_dataset)

English
Unique labels: [0 1 2 3 4 5]

Finnish
Unique labels: [ 6  7  8  9 10 11]


Next, we combine the English and Finnish datasets. The final datasets are named `train_dataset`, `validation_dataset` and `test_dataset`.  

`train_dataset` is unbatched, shuffled with a seed, and batched again with a batch size of 32. The shuffle buffer is 1000, which is larger than the training dataset so the shuffle is complete. Shuffling mixes English and Finnish samples, so every batch contains classes from both languages. This helps the model learn better. The seed makes the shuffle reproducible. 

In [7]:
train_dataset = en_train_dataset.concatenate(fi_train_dataset).unbatch().shuffle(1000, seed=29).batch(32)
validation_dataset = en_validation_dataset.concatenate(fi_validation_dataset)
test_dataset = en_test_dataset.concatenate(fi_test_dataset)

Let's print all the indices and their corresponding class names.

In [8]:
class_names = en_class_names + fi_class_names

# check that indices are correct
for i, name in enumerate(class_names):
    print(i, "=", name)

0 = down
1 = left
2 = right
3 = start
4 = stop
5 = up
6 = alas
7 = oikea
8 = paalle
9 = pysahdy
10 = vasen
11 = ylos


One more check for each final dataset.

In [9]:
print("Train")
check_labels(train_dataset)
print("\nValidation")
check_labels(validation_dataset)
print("\nTest")
check_labels(test_dataset)

Train
Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11]

Validation
Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11]

Test
Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11]


Now we have indices from 0 to 11 in all of the three final datasets. 

<h2>4. Modeling</h2>

 - Valitaan koneoppimismenetelmä, opetetaan malli ja validoidaan se (validation).
 - Dokumentointiin on sisällytettävä tarkasti valittu menetelmä, käytetyt parametrit sekä mitattu mallin suorituskyky.
 - Millä perusteella tehty valinnat
 - Taulukko on hyvä viestimiseen
 - Ei tarvitse olla yksityiskohtainen, mutta selitetään miksi on päädytty lopputulokseen.

<h2>5. Evaluation</h2>

 - Arvioidaan mallin todellinen onnistuminen.
 - Selvitetään, kuinka hyvin malli vastaa sille alun perin asetettuihin liiketoimintavaatimuksiin (business requirements).<br>
 - Ei katsota mallin suorituskykyä (tämä tehdään kappaleessa 4), vaan palataan takaisin kappaleeseen 1. ja millä tavalla saavutetut tulokset vertautuu alkuperäisiin tavoitteisiin.

<h2>6. Deployment</h2>

- Malli viedään käytäntöön (deployment).
- Tähän sisältyy tulosten viestiminen ja suositusten luominen siitä, miten mallia tulisi hyödyntää käytännössä tai mitä jatkotoimenpiteitä tulisi tehdä.